In [10]:
import pandas as pd
from datetime import datetime, timedelta
from pathlib import Path

data_dir = Path("/home/carl-wanninger/data/")

In [11]:
# === Parameters ===
YEAR = 2016
p1 = data_dir / "opfingen" / "storage_units-plugged_in.csv"
p2 = data_dir / "opfingen" / "storage_units-soc_departure_min_percent.csv"
p3 = data_dir / "opfingen" / "storage_units-soc_departure_max_percent.csv"

# === Step 1: Load Data ===
df1 = pd.read_csv(p1, index_col=0)
df2 = pd.read_csv(p2, index_col=0)
df3 = pd.read_csv(p3, index_col=0)

# === Step 2: Create datetime index ===
start_time = pd.Timestamp(f'{YEAR}-01-01 00:00')
date_range = pd.date_range(start=start_time, periods=len(df1), freq='15min')
df1.index = date_range
df2.index = date_range
df3.index = date_range

# === Step 3: Extract Sessions ===
records = []

In [13]:
for charger_id in df1.columns:
    series = df1[charger_id]
    active = series.dropna()
    # Group by EV session ID
    for ev_id, group in active.groupby(active):
        start_time = group.index[0]
        end_time = group.index[-1]
        
        start_soc = df2.at[start_time, charger_id] if pd.notna(df2.at[start_time, charger_id]) else None
        target_soc = df3.at[end_time, charger_id] if pd.notna(df3.at[end_time, charger_id]) else None

        record = {
            'ChargerID': charger_id,
            'EVID': f"{charger_id}_ev",
            'StartOfProcess': start_time,
            'EndOfProcess': end_time,
            'StartSoc': start_soc,
            'TargetSoc': target_soc
        }
        
        
        if target_soc is None or start_soc is None:
            print(record)
        elif target_soc > start_soc:
            records.append(record)

# === Step 4: Create Final DataFrame ===
result_df = pd.DataFrame(records)
result_df = result_df.sort_values(by=['StartOfProcess', 'ChargerID']).reset_index(drop=True)

# === Optional: Save to CSV ===
result_path = data_dir / "ev" / f"charging_sessions_{YEAR}.csv"
result_df.to_csv(result_path, index=False)

# Display

{'ChargerID': 'charger_1', 'EVID': 'charger_1_ev', 'StartOfProcess': Timestamp('2016-02-15 17:30:00'), 'EndOfProcess': Timestamp('2016-02-15 18:45:00'), 'StartSoc': 36.9075226068287, 'TargetSoc': None}
{'ChargerID': 'charger_1', 'EVID': 'charger_1_ev', 'StartOfProcess': Timestamp('2016-05-11 13:30:00'), 'EndOfProcess': Timestamp('2016-05-11 14:15:00'), 'StartSoc': 48.3940804731272, 'TargetSoc': None}
{'ChargerID': 'charger_1', 'EVID': 'charger_1_ev', 'StartOfProcess': Timestamp('2016-07-14 16:00:00'), 'EndOfProcess': Timestamp('2016-07-14 17:00:00'), 'StartSoc': 32.4178078655841, 'TargetSoc': None}
{'ChargerID': 'charger_2', 'EVID': 'charger_2_ev', 'StartOfProcess': Timestamp('2016-12-12 08:30:00'), 'EndOfProcess': Timestamp('2016-12-12 09:30:00'), 'StartSoc': 45.0920832116813, 'TargetSoc': None}
{'ChargerID': 'charger_3', 'EVID': 'charger_3_ev', 'StartOfProcess': Timestamp('2016-03-02 18:15:00'), 'EndOfProcess': Timestamp('2016-03-02 19:00:00'), 'StartSoc': 31.605635612613, 'TargetSoc